# Clean and standardise cell type annotations

In [51]:
suppressPackageStartupMessages({
  library(tidyverse)
})

# Load data

In [ ]:
files <- list.files(
  path = "../results/tables",
  pattern = "^cluster_annotation_final.tsv$",
  recursive = TRUE,
  full.names = TRUE
)

In [53]:
cluster.annotation <- lapply(
  files,
  function(x) {
    read.table(
      file = x,
      sep = "\t",
      header = TRUE,
      row.names = 1,
      stringsAsFactors = FALSE
    )
  }
) %>%
  do.call(rbind, .) %>%
  distinct() %>%
  select(cluster_main, cluster_coarse, cluster_fine) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_macrophage", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        c("B_naive_transitional", "B_naive") ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        c("DC_conventional_1", "DC_conventional_2") ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_macrophage", "Mono_CD14_CD16") ~ "Mono_CD14",
        c("Mono_CD16", "Mono_CD16_IFN") ~ "Mono_CD16",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN") ~ "T_CD4_naive",
        c("T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector") ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        c("T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN") ~ "T_CD8_naive",
        c("T_CD8_memory_central", "T_CD8_memory_effector") ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        c("B_naive", "B_memory", "B_plasma") ~ "B",
        c("DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid") ~ "DC",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD16") ~ "Mono",
        c("NK_CD56bright", "NK_CD56dim") ~ "NK",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c(
          "T_CD4_naive", "T_CD4_memory", "T_regulatory_naive", "T_regulatory_memory",
          "T_CD8_naive", "T_CD8_memory", "T_MAIT", "T_GD", "T_DN"
        ) ~ "T"
      ),
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  ) %>%
  arrange(cluster_main, cluster_coarse, cluster_fine) %>%
  filter(cluster_main %in% c("B", "DC", "ILC", "Mono", "NK", "T")) %>%
  droplevels()
rownames(cluster.annotation) <- NULL

# Clean and standardise names

In [54]:
cluster.annotation <- cluster.annotation %>%
  mutate(
    cluster_main_full = case_match(
      cluster_main,
      "B" ~ "B cell",
      "DC" ~ "Dendritic cell",
      "ILC" ~ "Innate lymphoid cell",
      "Mono" ~ "Monocyte",
      "NK" ~ "Natural killer cell",
      "T" ~ "T cells"
    ),
    cluster_main_short = case_match(
      cluster_main,
      "B" ~ "B",
      "DC" ~ "DC",
      "ILC" ~ "ILC",
      "Mono" ~ "Mono",
      "NK" ~ "NK",
      "T" ~ "T"
    ),
    cluster_coarse_full = case_match(
      cluster_coarse,
      "B_naive" ~ "Naive B cell",
      "B_memory" ~ "Memory B cell",
      "B_plasma" ~ "Plasma cell",
      "DC_AXL_SIGLEC6" ~ "AXL+SIGLEC6+ dendtritic cell",
      "DC_conventional" ~ "Conventional dendritic cell",
      "DC_plasmacytoid" ~ "Plasmacytoid dendritic cell",
      "ILC" ~ "Innate lymphoid cell",
      "Mono_CD14" ~ "CD14+ monocyte",
      "Mono_CD16" ~ "CD16+ monocyte",
      "NK_CD56bright" ~ "CD56bright NK cell",
      "NK_CD56dim" ~ "CD56dim NK cell",
      "T_CD4_naive" ~ "Naive CD4+ T cell",
      "T_CD4_memory" ~ "Memory CD4+ T cell",
      "T_regulatory_naive" ~ "Naive Regulatory T cell",
      "T_regulatory_memory" ~ "Memory Regulatory T cell",
      "T_CD8_naive" ~ "Naive CD8+ T cell",
      "T_CD8_memory" ~ "Memory CD8+ T cell",
      "T_MAIT" ~ "Mucosal-associated invariant T cell",
      "T_GD" ~ "Gamma delta T cell",
      "T_DN" ~ "Double negative T cell"
    ),
    cluster_coarse_short = case_match(
      cluster_coarse,
      "B_naive" ~ "Naive",
      "B_memory" ~ "Memory",
      "B_plasma" ~ "Plasma",
      "DC_AXL_SIGLEC6" ~ "AXL+SIGLEC6+",
      "DC_conventional" ~ "Conventional",
      "DC_plasmacytoid" ~ "Plasmacytoid",
      "ILC" ~ "ILC",
      "Mono_CD14" ~ "CD14+",
      "Mono_CD16" ~ "CD16+",
      "NK_CD56bright" ~ "CD56bright",
      "NK_CD56dim" ~ "CD56dim",
      "T_CD4_naive" ~ "CD4+ Naive",
      "T_CD4_memory" ~ "CD4+ Memory",
      "T_regulatory_naive" ~ "Regulatory Naive",
      "T_regulatory_memory" ~ "Regulatory Memory",
      "T_CD8_naive" ~ "CD8+ Naive",
      "T_CD8_memory" ~ "CD8+ Memory",
      "T_MAIT" ~ "MAIT",
      "T_GD" ~ "\u03B3\u03B4",
      "T_DN" ~ "CD4-CD8-"
    ),
    cluster_fine_full = case_match(
      cluster_fine,
      "B_naive_transitional" ~ "Transitional naive B cell",
      "B_naive" ~ "Naive B cell",
      "B_memory" ~ "Memory B cell",
      "B_plasma" ~ "Plasma cell",
      "DC_AXL_SIGLEC6" ~ "AXL+SIGLEC6+ dendtritic cell",
      "DC_conventional_1" ~ "Type 1 conventional dendritic cell",
      "DC_conventional_2" ~ "Type 2 conventional dendritic cell",
      "DC_plasmacytoid" ~ "Plasmacytoid dendritic cell",
      "ILC" ~ "Innate lymphoid cell",
      "Mono_CD14" ~ "CD14+ monocyte",
      "Mono_CD14_platelet" ~ "CD14+ monocyte-platelet aggregate",
      "Mono_CD14_IL1B" ~ "IL1Bhi CD14+ monocyte",
      "Mono_CD14_IFN" ~ "Interferon-signalling CD14+ monocyte",
      "Mono_CD14_macrophage" ~ "Macrophage-like CD14+ monocyte",
      "Mono_CD14_CD16" ~ "CD14+CD16+ intermediate monocyte",
      "Mono_CD16" ~ "CD16+ monocyte",
      "Mono_CD16_IFN" ~ "Interferon-signalling CD16+ monocyte",
      "NK_CD56bright" ~ "CD56bright NK cell",
      "NK_CD56dim" ~ "CD56dim NK cell",
      "T_CD4_naive" ~ "Naive CD4+ T cell",
      "T_CD4_naive_SOX4" ~ "SOX4hi Naive CD4+ T cell",
      "T_CD4_naive_IFN" ~ "Interferon-signalling Naive CD4+ T cell",
      "T_CD4_memory_central" ~ "Central memory CD4+ T cell",
      "T_CD4_memory_central_IFN" ~ "Interferon-signalling central memory CD4+ T cell",
      "T_CD4_memory_effector" ~ "Effector memory CD4+ T cell",
      "T_regulatory_naive" ~ "Naive Regulatory T cell",
      "T_regulatory_memory" ~ "Memory Regulatory T cell",
      "T_CD8_naive" ~ "Naive CD8+ T cell",
      "T_CD8_naive_SOX4" ~ "SOX4hi Naive CD8+ T cell",
      "T_CD8_naive_IFN" ~ "Interferon-signalling Naive CD8+ T cell",
      "T_CD8_memory_central" ~ "Central memory CD8+ T cell",
      "T_CD8_memory_effector" ~ "Effector memory CD8+ T cell",
      "T_MAIT" ~ "Mucosal-associated invariant T cell",
      "T_GD" ~ "Gamma delta T cell",
      "T_DN" ~ "Double negative T cell"
    ),
    cluster_fine_short = case_match(
      cluster_fine,
      "B_naive_transitional" ~ "Transitional",
      "B_naive" ~ "Naive",
      "B_memory" ~ "Memory",
      "B_plasma" ~ "Plasma",
      "DC_AXL_SIGLEC6" ~ "AXL+SIGLEC6+",
      "DC_conventional_1" ~ "Conventional 1",
      "DC_conventional_2" ~ "Conventional 2",
      "DC_plasmacytoid" ~ "Plasmacytoid",
      "ILC" ~ "ILC",
      "Mono_CD14" ~ "CD14+",
      "Mono_CD14_platelet" ~ "CD14+-Plt",
      "Mono_CD14_IL1B" ~ "CD14+ IL1Bhi",
      "Mono_CD14_IFN" ~ "CD14+ IFN",
      "Mono_CD14_macrophage" ~ "CD14+ Macro",
      "Mono_CD14_CD16" ~ "CD14+CD16+",
      "Mono_CD16" ~ "CD16+",
      "Mono_CD16_IFN" ~ "CD16+ IFN",
      "NK_CD56bright" ~ "CD56bright",
      "NK_CD56dim" ~ "CD56dim",
      "T_CD4_naive" ~ "CD4+ Naive",
      "T_CD4_naive_SOX4" ~ "CD4+ Naive SOX4hi",
      "T_CD4_naive_IFN" ~ "CD4+ Naive IFN",
      "T_CD4_memory_central" ~ "CD4+ CM",
      "T_CD4_memory_central_IFN" ~ "CD4+ CM IFN",
      "T_CD4_memory_effector" ~ "CD4+ EM",
      "T_regulatory_naive" ~ "Regulatory Naive",
      "T_regulatory_memory" ~ "Regulatory Memory",
      "T_CD8_naive" ~ "CD8+ Naive",
      "T_CD8_naive_SOX4" ~ "CD8+ Naive SOX4hi",
      "T_CD8_naive_IFN" ~ "CD8+ Naive IFN",
      "T_CD8_memory_central" ~ "CD8+ CM",
      "T_CD8_memory_effector" ~ "CD8+ EM",
      "T_MAIT" ~ "MAIT",
      "T_GD" ~ "\u03B3\u03B4",
      "T_DN" ~ "CD4-CD8-"
    )
  ) %>% 
  select(
    starts_with("cluster_main"),
    starts_with("cluster_coarse"),
    starts_with("cluster_fine")
  )

In [ ]:
openxlsx::write.xlsx(
  cluster.annotation,
  file = "/project/fuggerlab/rfarooq/project/ocrelizumab_multiome/results/tables/cite_seq/cohort_treatment_naive/cluster_annotate/cluster_annotation_final_clean.xlsx",
  overwrite = TRUE
)

# Session info

In [56]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.3 forcats_1.0.0   stringr_1.5.1   dplyr_1.1.4    
 [5] purrr_1.0.2     readr_2.1.5     tidyr_1.3.1     tibble_3.2.1   
 [9] ggplot2_3.5.1   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] gtable_0.3.5      jsonlite_1.8.9    compiler_4.3.3    crayon_1.5.3     
 [5] zip_2.3.1         Rcpp_1.0.13       tidyselect_1.2.1  IRdisplay_1.1    
 [9] scales_1.3.0      uuid_1.2-1        fastmap_1.2.0     IRkernel_1.3.2   
[13] R6_2.5.1          generics_0.1.3    openxlsx_4.2.7.1  munsell_0.5.1    
[17] pillar_1.9.0      tzdb_0